In [8]:
from pwn import *

In [64]:
HOST = "34.47.176.25"
PORT = 4901

def test(io):
    io.recvuntil(b"> ")
    io.sendline(b"2")

    io.recvuntil(b"Transmission: ")
    io.sendline(b"AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA")

    io.recvuntil(b"Ciphertext:\n")
    ct = io.recvline().strip().decode()
    return bytes.fromhex(ct)

io = remote(HOST, PORT)

ct = test(io)
blocks = [ct[i:i+16] for i in range(0,len(ct),16)]
print(f"ciphertext: {ct}")
print(f"blocks    : {blocks}")
if len(blocks) != len(set(blocks)):
    print("ecb, good...")
io.close()

ciphertext: b'\xac\x86\x87\x01[\xdaT\x95[U\x8c\x1a\xc5h\xc8]\xac\x86\x87\x01[\xdaT\x95[U\x8c\x1a\xc5h\xc8]\xd9\xdf\x9c9\xa5\xcd\x0bR?}\xc9\xc6<\xe1h\xb2'
blocks    : [b'\xac\x86\x87\x01[\xdaT\x95[U\x8c\x1a\xc5h\xc8]', b'\xac\x86\x87\x01[\xdaT\x95[U\x8c\x1a\xc5h\xc8]', b'\xd9\xdf\x9c9\xa5\xcd\x0bR?}\xc9\xc6<\xe1h\xb2']
ecb, good...


In [61]:
def get_ticket(username: bytes):
    io.recvuntil(b"> ")
    io.sendline(b"1")
    
    io.recvuntil(b"Username: ")
    io.sendline(username)
    
    io.recvuntil(b"Encrypted ticket:\n")
    ct = io.recvline().strip().decode()
    return ct

def usr_spoof(emperor: bytes):
    io.recvuntil(b"> ")
    io.sendline(b"2")

    io.recvuntil(b"Transmission: ")
    io.sendline(emperor)

    io.recvuntil(b"Ciphertext:\n")
    ct = io.recvline().strip().decode()
    return ct

def forged_ticket(ticket: bytes, emperor: bytes):
    payload = ticket + emperor
    
    io.recvuntil(b"> ")
    io.sendline(b"3")

    io.recvuntil(b"Encrypted ticket: ")
    io.sendline(payload.hex().encode())
    io.recvuntil(b"Access granted.\n")
    io.recvline()
    output = io.recvline()
    return output.strip().decode()

In [60]:
def find_flag(text: str):
    match = re.search(r"ACE\{.*?\}", text)
    return match.group(0) if match else None

In [ ]:
HOST = "34.47.176.25"
PORT = 4901
# context.log_level = 'debug'
context.log_level = 'error'

io = remote(HOST, PORT)
username = b"AAAAA"
payload = b"emperor"

clean_ticket = get_ticket(username)
print(f"size length: {len(clean_ticket)//2}")
print(clean_ticket)

spoof_ticket = clean_ticket[:16*4]
print(f"size length: {len(spoof_ticket)//2}")
print(spoof_ticket)


emperor = usr_spoof(payload)
print(f"\nsize length: {len(emperor)//2}")
print(emperor)

result = forged_ticket(bytes.fromhex(spoof_ticket),bytes.fromhex(emperor))
flag = find_flag(result)
print(f"Flag: {flag}")
io.close()

size length: 48
2f3e70c522d5d6a10879ab969dfdb06b2e5fe1e7361341d96fe45e7477d1f0a0982c7a9346b489bb5c61cff0a59dce60
size length: 32
2f3e70c522d5d6a10879ab969dfdb06b2e5fe1e7361341d96fe45e7477d1f0a0

size length: 16
8288972ea0904c6a7438675d35054a60
Flag: ACE{Long_live_the_Empire_0a907347c304f052db464b9b1991b259}
